In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import sqlite3

from src.config import DATABASE_FILE

from src.quality.quality_checks import (
    load_quality_enabled_datasets,
    run_quality_checks,
)

from src.quality.quality_metrics import (
    calculate_quality_metrics,
)

# Phase 4 - Data Quality Framework

## Objectif

L'objectif de cette phase est de mettre en place un framework de contrôle
qualité permettant d'évaluer la fiabilité des données intégrées dans la
plateforme d'investissement.

La qualité des données constitue un élément fondamental dans les institutions
financières. Une donnée incorrecte peut conduire à :

- une analyse erronée ;
- une décision d'investissement inappropriée ;
- une perte financière ;
- un risque opérationnel ;
- un risque de conformité.

Cette phase introduit un modèle de contrôle qualité standardisé,
intégrable à tout nouveau dataset chargé dans la plateforme.

## Architecture du Framework

Le framework qualité repose sur quatre composants principaux :

1. dataset_registry
2. quality_checks.py
3. quality_metrics.py
4. data_control_center.py

Flux de traitement :

Source Registry
↓
Database Loader
↓
SQLite Database
↓
Dataset Registry
↓
Quality Checks
↓
Quality Metrics
↓
Data Control Center

## Datasets soumis au contrôle qualité

Le framework identifie automatiquement les datasets dont :

- active = 1
- quality_enabled = 1

Les règles qualité sont donc pilotées par le Dataset Registry
et non par des noms de tables codés en dur.

In [9]:
connection = sqlite3.connect(
    DATABASE_FILE
)

datasets = load_quality_enabled_datasets(
    connection
)

datasets

,dataset_id,source_id,table_name,chronology_enabled
0,DS-002,SRC-002,securities_master,0


## Contrôles implémentés

QC-001 Table Availability

Vérifie que la table physique existe dans SQLite.

QC-002 Dataset Not Empty

Vérifie que le dataset contient au moins un enregistrement.

QC-003 Completeness

Vérifie que les champs obligatoires sont renseignés.

QC-004 Duplicate Rows

Détecte les doublons complets.

QC-005 Blank Values

Détecte les chaînes vides.

QC-006 Column Integrity

Détecte les colonnes dupliquées ou invalides.

QC-007 Chronology Readiness

Vérifie la présence de colonnes de type date lorsque
la chronologie est activée.


In [10]:
from src.quality.quality_checks import (
    run_quality_checks,
)

controls = run_quality_checks()

controls

,dataset_id,source_id,table_name,control_id,control_name,status,issues,records_checked,execution_date,notes
0,DS-002,SRC-002,securities_master,QC-001,Table Availability,PASS,0,0,2026-09-15 16:28:18,Physical table existence
1,DS-002,SRC-002,securities_master,QC-002,Dataset Not Empty,PASS,0,103,2026-09-15 16:28:18,Dataset contains records
2,DS-002,SRC-002,securities_master,QC-003,Completeness,PASS,0,103,2026-09-15 16:28:18,Mandatory field completeness
3,DS-002,SRC-002,securities_master,QC-004,Duplicate Rows,PASS,0,103,2026-09-15 16:28:18,Duplicate row detection
4,DS-002,SRC-002,securities_master,QC-005,Blank Values,PASS,0,103,2026-09-15 16:28:18,Blank value detection
5,DS-002,SRC-002,securities_master,QC-006,Column Integrity,PASS,0,103,2026-09-15 16:28:18,Column structure validation
6,DS-002,SRC-002,securities_master,QC-007,Chronology Readiness,NOT_APPLICABLE,0,103,2026-09-15 16:28:18,Chronology disabled


## KPI calculés

Le framework produit les indicateurs suivants :

- Availability Rate
- Completeness Rate
- Uniqueness Rate
- Freshness Rate
- Chronology Rate
- Global Quality Score
- Global Quality Grade
- Global Status

## Modèle de scoring

Chaque métrique contribue au score global via une pondération.

Pondérations :

- Availability Rate : 20%
- Completeness Rate : 35%
- Uniqueness Rate : 20%
- Freshness Rate : 15%
- Chronology Rate : 10%

In [11]:
connection = sqlite3.connect(
    DATABASE_FILE
)

dataframe = pd.read_sql(
    """
    SELECT *
    FROM securities_master
    """,
    connection,
)

connection.close()

quality_results = (
    calculate_quality_metrics(
        dataframe=dataframe,
        chronology_enabled=False,
    )
)

quality_results["metrics"]

,metric,value,status,threshold,records_checked,execution_date
0,Availability Rate,100.0,PASS,95.0,103,2026-09-15 16:28:22
1,Completeness Rate,100.0,PASS,95.0,103,2026-09-15 16:28:22
2,Uniqueness Rate,100.0,PASS,95.0,103,2026-09-15 16:28:22
3,Freshness Rate,100.0,PASS,95.0,103,2026-09-15 16:28:22
4,Chronology Rate,100.0,PASS,95.0,103,2026-09-15 16:28:22


In [12]:
print(
    "Score:",
    quality_results[
        "global_quality_score"
    ]
)

print(
    "Grade:",
    quality_results[
        "global_quality_grade"
    ]
)

print(
    "Status:",
    quality_results[
        "global_quality_status"
    ]
)

Score: 100.0
Grade: A
Status: PASS


## Gestion des champs optionnels

Certaines colonnes ne sont pas obligatoires.

Exemple :

reason

La colonne reason est utilisée pour documenter
les échecs de matching.

Lorsque le statut est MATCH, une valeur NULL
est attendue.

Le framework permet donc de définir des colonnes
optionnelles via :

OPTIONAL_COLUMNS

afin d'éviter les faux positifs qualité.

## Dashboard Data Quality

Le framework génère automatiquement :

audit/data_control_center.xlsx

Le dashboard contient :

- Executive Summary
- Quality Controls
- Quality Issues
- Quality History

## Conclusion

La Phase 4 a permis la mise en place d'un framework qualité
générique et extensible.

Principales réalisations :

- Contrôles qualité automatisés
- KPI qualité standardisés
- Modèle de scoring pondéré
- Dashboard Excel automatisé
- Architecture pilotée par Dataset Registry
- Support des champs optionnels
- Préparation de l'intégration future avec
  quality_run,
  quality_control_result,
  quality_metric_result

Le framework est désormais capable de contrôler
automatiquement tout nouveau dataset activé dans
le Dataset Registry.

Future Enhancements
 
- Dataset-specific quality rules
- Freshness monitoring based on acquisition dates
- Persistence into quality_run tables
- Multi-dataset quality scoring
- Extended chronology validation